# Figure: Negative Log LUT

Left: the normalised negative-log look-up table `L(x) = -log(x) / -log(ε)`.  
Right: the LUT applied to a recovered transmission map from the experiment results.

This is the first operator in the forward pipeline: `txm → L(txm) → windowing → ...`

**Requires:** `nb_run.ipynb`

Output: `build/negative_log_lut.png`

In [ ]:
%%capture
import os, subprocess, sys

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    DRIVE_ROOT = "/content/drive/MyDrive/thesis"  # <-- edit if your Drive path differs

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", f"{DRIVE_ROOT}/notebooks/requirements.txt",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "-e", DRIVE_ROOT,
    ])

    # Change to figures/ so load_results() can find the cached data
    os.chdir(f"{DRIVE_ROOT}/figures")


In [ ]:
import sys, os, glob, joblib

import numpy as np
import matplotlib.pyplot as plt
from chest_xray_sim.inverse import operators as ops

def load_results(results_dir="results"):
    hits = sorted(glob.glob(f"{results_dir}/*/imgs.joblib"))
    if not hits:
        raise FileNotFoundError(f"No results in '{results_dir}'. Run nb_run.ipynb first.")
    d = os.path.dirname(hits[-1])
    print(f"Loading from {d}")
    return {"txm": joblib.load(f"{d}/txm.joblib")}

In [ ]:
txm = load_results()["txm"]
sample_txm = txm[0]  # first recovered transmission map

In [ ]:
values = np.linspace(1e-6, 1.0, 500)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(values, ops.negative_log(values), color="steelblue")
ax[0].set_title("Negative Log LUT: $L(x) = -\\log(x)\\,/\\,-\\log(\\varepsilon)$")
ax[0].set_xlabel("Transmission (input)")
ax[0].set_ylabel("Normalised output")
ax[0].set_xlim(0, 1)
ax[0].set_ylim(0, 1)
ax[0].grid(True, alpha=0.3)

ax[1].imshow(ops.negative_log(sample_txm), cmap="gray", vmin=0, vmax=1)
ax[1].set_title("neg-log applied to recovered TXM")
ax[1].axis("off")

fig.tight_layout()
fig.savefig("build/negative_log_lut.png", bbox_inches="tight")
plt.show()